# Семинар 09. HTTP и JSON


## Цели

После семинара вы сможете:

- разбирать URL, HTTP-запрос и HTTP-ответ;
- интерпретировать коды состояния и заголовок `Content-Type`;
- выбирать HTTP-метод с учётом безопасности и идемпотентности;
- сериализовать JSON и надёжно обрабатывать сетевые ответы.

> **Формат:** справочный материал для индивидуального проекта. Отдельного задания по семинару нет.

## Перед началом

Установите `requests`. Для сетевых примеров требуется подключение к интернету.


## Полезные ссылки

- [RFC 9110: HTTP Semantics](https://www.rfc-editor.org/rfc/rfc9110.html)
- [RFC 8259: JSON](https://www.rfc-editor.org/rfc/rfc8259.html)
- [Requests: Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)

## Протокол HTTP

HTTP — прикладной протокол обмена сообщениями по модели «запрос — ответ». Клиент указывает намерение с помощью метода и адресует ресурс, а сервер возвращает ответ с кодом состояния, заголовками и, возможно, содержимым. Протокол не требует, чтобы ресурс был файлом: это может быть пользователь, заказ, вычисление или коллекция объектов.

Разберём адрес `https://api.example.com/v1/search?text=python&limit=10#results`:

| Часть | Значение | Назначение |
|---|---|---|
| схема | `https` | HTTP через защищённое TLS-соединение |
| хост | `api.example.com` | имя сервера |
| путь | `/v1/search` | идентификатор ресурса на сервере |
| query string | `text=python&limit=10` | параметры запроса |
| fragment | `results` | локальная часть ссылки; браузер не отправляет её серверу в HTTP-запросе |

HTTPS шифрует трафик и позволяет проверить подлинность сервера по сертификату. Это защищает данные в пути, но само по себе не делает API или полученный ответ доверенным.

HTTP-запрос содержит:

- метод, например `GET` или `POST`;
- целевой путь и параметры;
- заголовки с метаданными, например `Accept`, `Content-Type` и `Authorization`;
- необязательное содержимое запроса.

HTTP по своей семантике не хранит состояние между запросами. Если приложению нужны пользовательская сессия или авторизация, оно передаёт идентификатор в cookie либо заголовке и связывает его с состоянием на своей стороне.

В Python есть встроенный пакет `urllib`, но в семинаре используется сторонняя синхронная библиотека `requests`. В асинхронном приложении её прямой вызов заблокирует event loop; для такого приложения нужен асинхронный HTTP-клиент либо явный перенос блокирующего вызова в поток.


In [ ]:
import requests


response = None
try:
    response = requests.get(
        "https://example.com",
        params={"source": "seminar"},
        headers={"Accept": "text/html"},
        timeout=(3.05, 10),  # тайм-ауты подключения и чтения
    )
    response.raise_for_status()
    print("Final URL:", response.url)
    print("Status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
except requests.RequestException as error:
    print(f"Request failed: {error}")


## HTTP-ответ

Объект `Response` содержит код состояния, заголовки, итоговый URL после перенаправлений и содержимое ответа. Трёхзначные коды состояния делятся на пять классов:

| Класс | Значение | Примеры |
|---|---|---|
| `1xx` | промежуточная информация | запрос принят, обработка продолжается |
| `2xx` | запрос успешно обработан | `200 OK`, `201 Created`, `204 No Content` |
| `3xx` | для завершения нужны дополнительные действия | перенаправление или использование кэша |
| `4xx` | запрос не может быть выполнен в текущем виде | `400 Bad Request`, `401 Unauthorized`, `404 Not Found`, `429 Too Many Requests` |
| `5xx` | сервер не смог выполнить допустимый запрос | `500 Internal Server Error`, `503 Service Unavailable` |

`response.raise_for_status()` выбрасывает `HTTPError` для ответов 4xx и 5xx. Тело ответа при этом всё равно может содержать полезное описание ошибки.

Способ чтения зависит от `Content-Type` и контракта API:

- `response.text` декодирует тело как текст;
- `response.content` возвращает байты, например для изображения;
- `response.json()` разбирает JSON, но выбрасывает исключение для пустого или некорректного тела.

Успешный разбор JSON не означает успешный HTTP-ответ: сервер может вернуть JSON с деталями ошибки и кодом 4xx или 5xx. Поэтому сначала проверяют статус, затем формат и структуру данных.


In [ ]:
if response is not None:
    content_type = response.headers.get("Content-Type", "")
    if content_type.startswith("text/"):
        print(response.text[:500])  # выводим только начало ответа


## Формат JSON

JSON (JavaScript Object Notation) — текстовый формат обмена структурированными данными. JSON-значением может быть объект, массив, строка, число, `true`, `false` или `null`; эти значения можно вкладывать друг в друга.

| JSON | Python после `json.loads()` |
|---|---|
| object | `dict` |
| array | `list` |
| string | `str` |
| number без точки и экспоненты | `int` |
| number с точкой или экспонентой | `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

Имена полей объекта и строки записываются в двойных кавычках. Стандартный JSON не допускает комментарии и запятую после последнего элемента. Числа `NaN` и `Infinity` также не входят в стандарт JSON, хотя некоторые библиотеки принимают их как расширение.

Встроенный модуль `json` преобразует данные в строку через `dumps()` и обратно через `loads()`. Функции `dump()` и `load()` выполняют те же операции с файловым объектом. После разбора ответа всё равно нужно проверить ожидаемые типы и обязательные поля: синтаксически корректный JSON не обязательно соответствует контракту API.


In [ ]:
import json


data = {
    "name": "Иван",
    "age": 30,
    "skills": ["Python", "HTTP"],
    "active": True,
    "city": None,
}

dumped = json.dumps(data, ensure_ascii=False, indent=2)
print(dumped)

unpacked = json.loads(dumped)
assert unpacked == data


In [ ]:
import requests


# Метод json() десериализует тело ответа в объект Python.
try:
    response = requests.get(
        "https://jsonplaceholder.typicode.com/todos/1",
        headers={"Accept": "application/json"},
        timeout=(3.05, 10),
    )
    response.raise_for_status()
    media_type = response.headers.get("Content-Type", "").split(";", 1)[0]
    if media_type != "application/json" and not media_type.endswith("+json"):
        raise ValueError(f"Unexpected Content-Type: {media_type or 'missing'}")
    result = response.json()

    if not isinstance(result, dict):
        raise ValueError("Expected a JSON object")
    required_fields = {"userId", "id", "title", "completed"}
    missing_fields = required_fields - result.keys()
    if missing_fields:
        raise ValueError(f"Missing fields: {sorted(missing_fields)}")

    print(result["title"], "completed =", result["completed"])
except (requests.RequestException, ValueError) as error:
    print(f"Could not load JSON: {error}")


## HTTP-методы и их свойства

Метод сообщает серверу намерение клиента. Для корректной обработки повторов и кэширования важны три свойства:

- **Безопасность:** клиент не просит изменить состояние ресурса. Служебные побочные эффекты вроде записи в лог допустимы.
- **Идемпотентность:** ожидаемый эффект нескольких одинаковых запросов на сервере совпадает с эффектом одного запроса. Ответы при этом могут различаться, например первый `DELETE` вернёт 204, а повторный — 404.
- **Кэшируемость:** ответ разрешено сохранить и переиспользовать при выполнении условий протокола и заголовков кэширования.

| Метод | Типичное назначение | Безопасный | Идемпотентный |
|---|---|:---:|:---:|
| `GET` | получить представление ресурса | да | да |
| `HEAD` | получить те же заголовки, что для `GET`, без тела ответа | да | да |
| `POST` | передать данные на обработку, часто создать подчинённый ресурс | нет | нет в общем случае |
| `PUT` | создать или полностью заменить ресурс по известному адресу | нет | да |
| `PATCH` | частично изменить ресурс | нет | не гарантируется |
| `DELETE` | удалить ресурс | нет | да |

`GET` нельзя использовать для запрошенного клиентом изменения состояния: браузер, прокси или робот может предварительно загрузить, повторить или закэшировать такой запрос. Неидемпотентный запрос нельзя автоматически повторять, если приложение не знает, что первоначальная операция не была применена. Для критичных операций API может поддерживать ключ идемпотентности.

В `requests` query-параметры передают через `params`, а JSON-тело — через `json`. Во втором случае библиотека сериализует объект и устанавливает подходящий `Content-Type`:

```python
payload = {"title": "Новая запись", "published": False}
response = requests.post(
    "https://api.example.com/posts",
    json=payload,
    timeout=(3.05, 10),
)
response.raise_for_status()
```

## Надёжность и безопасность клиента

- Всегда задавайте тайм-аут подключения и чтения. В `requests` это не жёсткий предел длительности всего скачивания.
- Обрабатывайте сетевые исключения, неожиданные статусы, неверный формат и несоответствие данных контракту отдельно.
- Переиспользуйте `requests.Session`, если выполняете серию запросов к одному сервису.
- Не размещайте токены в исходном коде и query string: они могут попасть в историю и журналы. Секрет обычно читают из окружения и передают через заголовок `Authorization` по HTTPS.
- Не записывайте в журнал пароли, токены и полные ответы с персональными данными.


## Где пригодится в индивидуальном проекте

HTTP понадобится, если проект предоставляет собственный API или обращается к внешнему сервису. Заранее зафиксируйте контракт каждого вызова: метод, путь, параметры, тело, успешные статусы и формат ошибки. Для входящих и исходящих данных проверяйте не только наличие JSON, но и его структуру.

Сетевой слой лучше отделить от бизнес-логики: одна функция отвечает за запрос, тайм-аут и разбор ответа, а другая принимает решение на основе уже проверенных данных. Это упрощает тестирование с подменёнными ответами и замену внешнего сервиса.


## Самопроверка

1. Какие части URL отправляются серверу, а какая часть остаётся в браузере?
2. Почему успешный вызов `response.json()` не доказывает, что запрос выполнен успешно?
3. Чем безопасность HTTP-метода отличается от идемпотентности?
4. Почему повтор одинакового `DELETE` может вернуть другой статус и всё же оставаться идемпотентным?
5. Чем параметры `params=` и `json=` в `requests` отличаются по расположению данных?
6. Какие ошибки должен обработать клиент внешнего API помимо статуса 5xx?


## Итоги

- HTTP описывает обмен запросами и ответами с ресурсами.
- Метод выражает намерение клиента, а статус сообщает результат обработки.
- Формат тела определяют заголовок `Content-Type` и контракт API.
- Корректный JSON нужно дополнительно проверять на ожидаемую структуру.
- Надёжный клиент задаёт тайм-ауты, обрабатывает ошибки и не раскрывает секреты.
